# SeqTrainer Titans paper-MAC — Stage B A100 capture

This notebook captures the complete strict Stage B A100 evidence bundle in Google Drive. Select **Runtime → Change runtime type → A100 GPU** before running it. A T4, L4, V100, or CPU is intentionally rejected.

The checked-out ref must include commit `c4c09f0` (or a descendant), which adds the pilot and this workflow. Do not continue if the preflight cell does not report an A100.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Change only these values when testing a different pushed branch or commit.
REPOSITORY_URL = 'https://github.com/Gonza10V/SeqTrainer.git'
REF = 'feat/titans-paper-mac-stage-b'  # must contain c4c09f0 or a descendant
WORKSPACE = '/content/SeqTrainer'
DRIVE_ROOT = '/content/drive/MyDrive/SeqTrainerA100'
RUN_ID = None  # None creates a UTC timestamped folder; set a unique label to override


In [ ]:
!nvidia-smi
import os, subprocess
os.makedirs(DRIVE_ROOT, exist_ok=True)
if not os.path.exists(WORKSPACE):
    subprocess.run(['git', 'clone', REPOSITORY_URL, WORKSPACE], check=True)
subprocess.run(['git', 'fetch', '--tags', 'origin'], cwd=WORKSPACE, check=True)
subprocess.run(['git', 'checkout', REF], cwd=WORKSPACE, check=True)
subprocess.run(['git', 'merge-base', '--is-ancestor', 'c4c09f0', 'HEAD'], cwd=WORKSPACE, check=True)
print('Checked-out commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=WORKSPACE, text=True).strip())


If checkout cannot find the ref, push the local branch containing `c4c09f0` first, or replace `REF` with an accessible commit. This notebook deliberately refuses to substitute an older checkout.

In [ ]:
%cd /content/SeqTrainer
!python -m pip install -q uv
!uv venv --system-site-packages .venv
# Keep Colab's CUDA-enabled torch from system site packages, but explicitly install SeqTrainer's non-Torch runtime dependencies.
!uv pip install --python .venv/bin/python --upgrade "numpy>=1.24,<2" "pandas>=1.5" "rdflib>=6.3.2" "scikit-learn>=1.3" "requests>=2.31" "sbol2>=1.4" "pytest>=8.0" "ruff>=0.4"
!uv pip install --python .venv/bin/python --no-deps -e .
!.venv/bin/python -c "import numpy, pandas, rdflib, requests, sbol2, sklearn, torch; import seqtrainer; print('SeqTrainer dependency imports: OK'); print('torch=', torch.__version__, 'cuda=', torch.version.cuda, 'device=', torch.cuda.get_device_name(0))"
!.venv/bin/python -m seqtrainer.torch.titans_paper_mac_stage_b.a100_pilot --preflight-only


In [ ]:
import shlex
from pathlib import Path
command = [
    '.venv/bin/python', 'scripts/run_titans_stage_b_a100_colab.py',
    '--repository-root', WORKSPACE,
    '--drive-root', DRIVE_ROOT,
    '--warmup-runs', '1', '--repetitions', '3',
]
if RUN_ID:
    command += ['--run-id', RUN_ID]
print(' '.join(shlex.quote(part) for part in command))
completed = subprocess.run(command, cwd=WORKSPACE, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(completed.stdout)
if completed.returncode:
    failed = sorted(Path(DRIVE_ROOT).glob('*/FAILED.txt'), key=lambda path: path.stat().st_mtime)
    if failed:
        debug_folder = failed[-1].parent
        print('Capture failed. Drive debug folder:', debug_folder)
        print((debug_folder / 'FAILED.txt').read_text())
        for log in sorted((debug_folder / 'logs').glob('*.txt')):
            tail = log.read_text(errors='replace').splitlines()[-120:]
            print(f'\n--- tail: {log.name} ---')
            print('\n'.join(tail))
    raise RuntimeError(f'Stage B A100 capture failed with exit code {completed.returncode}; see the output above.')


In [ ]:
from pathlib import Path
runs = sorted((path for path in Path(DRIVE_ROOT).iterdir() if (path / 'logs').exists()), key=lambda path: path.stat().st_mtime)
latest = runs[-1]
print('Latest A100 run folder:', latest)
if (latest / 'FAILED.txt').exists():
    print((latest / 'FAILED.txt').read_text())
    for log in sorted((latest / 'logs').glob('*.txt')):
        print(f'\n--- tail: {log.name} ---')
        print('\n'.join(log.read_text(errors='replace').splitlines()[-120:]))
else:
    print('Verified handoff folder:', latest)
    print('Share this folder (it contains a100/ and logs/), then send me its Drive link.')
    print((latest / 'README.txt').read_text())
